# DermaMNIST Model Training

## Setup and Reproducibility

!pip install -q "numpy==1.26.4" "scipy==1.13.1"

!pip install -q "transformers==4.44.2" --no-deps

!pip install -q "tokenizers>=0.19,<0.20" "huggingface-hub>=0.23,<0.25" "safetensors>=0.4"

!pip install -q "medmnist" "tf-keras"

In [ ]:
# SETUP AND REPRODUCIBILITY

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ['PYTHONHASHSEED'] = '42'

import random
import numpy as np
import tensorflow as tf

# Global reproducibility seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

# Imports
import medmnist
from medmnist import INFO
import gc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras import layers, models, applications, mixed_precision, callbacks
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from transformers import TFViTForImageClassification
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                              roc_auc_score, precision_score, recall_score, f1_score,
                              average_precision_score, log_loss)
from sklearn.calibration import calibration_curve
from scipy.signal import find_peaks
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')

print(f"NumPy: {np.__version__}")
print(f"TensorFlow: {tf.__version__}")
print(f"Random Seed: {SEED}")
print(f"Legacy Keras: {os.environ.get('TF_USE_LEGACY_KERAS')}")

2026-06-07 18:07:38.890573: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780855659.060220     134 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780855659.109558     134 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780855659.519298     134 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780855659.519336     134 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780855659.519339     134 computation_placer.cc:177] computation placer alr

NumPy: 1.26.4
TensorFlow: 2.19.0
Random Seed: 42
Legacy Keras: 1


## Configuration

In [ ]:
# CONFIGURATION

DATASET = 'dermamnist'
IMAGE_SIZE = 224
BATCH_SIZE = 4
EPOCHS = 30
LR_VIT = 2e-5
LR_CNN = 1e-4
DROPOUT_META = 0.5
META_BATCH_SIZE = 16
META_EPOCHS = 50
META_PATIENCE = 5
PATIENCE_EARLY = 7
PATIENCE_REDUCE = 3
REDUCE_FACTOR = 0.5
MIN_LR = 1e-6

OUTPUT_DIR = f"/kaggle/working/{DATASET}_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Mixed precision for speed
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print(f"Mixed Precision: {policy.name}")
print(f"Output Dir: {OUTPUT_DIR}")

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPUs will likely run quickly with dtype policy mixed_float16 as they all have compute capability of at least 7.0
Mixed Precision: mixed_float16
Output Dir: /kaggle/working/dermamnist_outputs


## Data Loading

In [ ]:
# DATA LOADER

def load_medmnist_data(dataset_flag, size=224, batch_size=8):
    """Loads MedMNIST v2 dataset and prepares tf.data pipelines."""
    info = INFO[dataset_flag]
    n_classes = len(info['label'])
    DataClass = getattr(medmnist, info['python_class'])

    print(f"Loading {dataset_flag}...")
    train_data = DataClass(split='train', download=True, size=size)
    val_data = DataClass(split='val', download=True, size=size)
    test_data = DataClass(split='test', download=True, size=size)

    X_train, y_train = train_data.imgs, train_data.labels
    X_val, y_val = val_data.imgs, val_data.labels
    X_test, y_test = test_data.imgs, test_data.labels

    # Convert grayscale to RGB if needed
    if X_train.shape[-1] != 3:
        X_train = np.repeat(X_train[..., np.newaxis], 3, -1)
        X_val = np.repeat(X_val[..., np.newaxis], 3, -1)
        X_test = np.repeat(X_test[..., np.newaxis], 3, -1)

    X_train = X_train.astype('float32')
    X_val = X_val.astype('float32')
    X_test = X_test.astype('float32')

    y_train_enc = to_categorical(y_train, n_classes)
    y_val_enc = to_categorical(y_val, n_classes)
    y_test_enc = to_categorical(y_test, n_classes)

    train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train_enc))\
        .shuffle(1000, seed=SEED).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val_enc)).batch(batch_size)
    test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test_enc)).batch(batch_size)

    return train_ds, val_ds, test_ds, n_classes, y_test, X_test, info

train_ds, val_ds, test_ds, n_classes, y_test_true, X_test_raw, dataset_info = \
    load_medmnist_data(DATASET, size=IMAGE_SIZE, batch_size=BATCH_SIZE)
y_true_flat = y_test_true.flatten()
y_true_cat = to_categorical(y_true_flat, n_classes)
CLASS_NAMES = [dataset_info['label'][str(i)] for i in range(n_classes)]
print(f"Classes: {n_classes}")
print(f"Class Names: {CLASS_NAMES}")
print(f"Test set size: {len(y_true_flat)}")

Loading dermamnist...


100%|██████████| 1.09G/1.09G [01:52<00:00, 9.71MB/s]  
I0000 00:00:1780855813.814745     134 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1780855813.820533     134 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Classes: 7
Class Names: ['actinic keratoses and intraepithelial carcinoma', 'basal cell carcinoma', 'benign keratosis-like lesions', 'dermatofibroma', 'melanoma', 'melanocytic nevi', 'vascular lesions']
Test set size: 2005


## Class Distribution Analysis

In [ ]:
# CLASS DISTRIBUTION ANALYSIS

DataClass = getattr(medmnist, dataset_info['python_class'])
train_labels = DataClass(split='train', size=IMAGE_SIZE).labels.flatten()
val_labels = DataClass(split='val', size=IMAGE_SIZE).labels.flatten()
test_labels = DataClass(split='test', size=IMAGE_SIZE).labels.flatten()

dist_rows = []
for i, name in enumerate(CLASS_NAMES):
    train_count = (train_labels == i).sum()
    val_count = (val_labels == i).sum()
    test_count = (test_labels == i).sum()
    total = train_count + val_count + test_count
    dist_rows.append({
        'Class ID': i,
        'Class Name': name,
        'Train': train_count,
        'Val': val_count,
        'Test': test_count,
        'Total': total,
        '% of Total': round(100 * total / (len(train_labels)+len(val_labels)+len(test_labels)), 2)
    })

dist_df = pd.DataFrame(dist_rows)
print("\n=== CLASS DISTRIBUTION TABLE ===")
print(dist_df.to_string(index=False))

ir = dist_df['Train'].max() / dist_df['Train'].min()
print(f"\nImbalance Ratio (IR): {ir:.2f}")
print(f"Total samples: Train={len(train_labels)}, Val={len(val_labels)}, Test={len(test_labels)}")
print(f"Missing values check: Train NaNs={pd.isna(train_labels).sum()}, Test NaNs={pd.isna(test_labels).sum()}")
print(f"Duplicates: handled by MedMNIST v2 official splits (no overlap between train/val/test)")

dist_df.to_csv(f"{OUTPUT_DIR}/class_distribution.csv", index=False)


=== CLASS DISTRIBUTION TABLE ===
 Class ID                                      Class Name  Train  Val  Test  Total  % of Total
        0 actinic keratoses and intraepithelial carcinoma    228   33    66    327        3.27
        1                            basal cell carcinoma    359   52   103    514        5.13
        2                   benign keratosis-like lesions    769  110   220   1099       10.97
        3                                  dermatofibroma     80   12    23    115        1.15
        4                                        melanoma    779  111   223   1113       11.11
        5                                melanocytic nevi   4693  671  1341   6705       66.95
        6                                vascular lesions     99   14    29    142        1.42

Imbalance Ratio (IR): 58.66
Total samples: Train=7007, Val=1003, Test=2005
Missing values check: Train NaNs=0, Test NaNs=0
Duplicates: handled by MedMNIST v2 official splits (no overlap between train/val/t

## Model Factory Function

In [ ]:
# MODEL FACTORY

def get_model(model_name, n_classes, input_shape=(224, 224, 3)):
    """Builds a model with the specified architecture."""
    inputs = layers.Input(shape=input_shape)

    if model_name == 'EfficientNetV2M':
        base = applications.EfficientNetV2M(include_top=False, weights='imagenet', input_tensor=inputs)
        x = layers.GlobalAveragePooling2D()(base.output)
        outputs = layers.Dense(n_classes, activation='softmax', dtype='float32')(x)

    elif model_name == 'InceptionResNetV2':
        x = layers.Rescaling(1./127.5, offset=-1)(inputs)
        base = applications.InceptionResNetV2(include_top=False, weights='imagenet', input_tensor=x)
        x = layers.GlobalAveragePooling2D()(base.output)
        outputs = layers.Dense(n_classes, activation='softmax', dtype='float32')(x)

    elif model_name == 'ConvNeXtBase':
        base = applications.ConvNeXtBase(include_top=False, weights='imagenet', input_tensor=inputs)
        x = layers.GlobalAveragePooling2D()(base.output)
        outputs = layers.Dense(n_classes, activation='softmax', dtype='float32')(x)

    elif model_name == 'ViT-Base':
        x = layers.Rescaling(1./127.5, offset=-1)(inputs)
        x = layers.Permute((3, 1, 2))(x)
        backbone = TFViTForImageClassification.from_pretrained(
            'google/vit-base-patch16-224', num_labels=n_classes,
            ignore_mismatched_sizes=True, use_safetensors=False)
        outputs = backbone.vit(x)[0]
        outputs = backbone.classifier(outputs[:, 0, :])
        outputs = layers.Activation('softmax', dtype='float32')(outputs)

    else:
        raise ValueError(f"Unknown Model: {model_name}")

    return models.Model(inputs=inputs, outputs=outputs, name=model_name)

## Trainer Function

In [ ]:
# TRAINER

def train_and_save(model_name):
    print(f"\n{'='*20} TRAINING {model_name} {'='*20}")
    tf.keras.backend.clear_session()
    gc.collect()

    model_path = f"{OUTPUT_DIR}/{model_name}_{DATASET}.keras"
    val_pred_path = f"{OUTPUT_DIR}/{model_name}_val_preds.npy"
    test_pred_path = f"{OUTPUT_DIR}/{model_name}_test_preds.npy"

    if os.path.exists(model_path):
        print(f"Loading existing model: {model_path}")
        model = get_model(model_name, n_classes)
        model.load_weights(model_path)
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    else:
        model = get_model(model_name, n_classes)
        lr = LR_VIT if model_name == 'ViT-Base' else LR_CNN
        model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                      loss='categorical_crossentropy', metrics=['accuracy'])

        cb = [
            ModelCheckpoint(model_path, save_best_only=True, monitor='val_accuracy', mode='max', verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=REDUCE_FACTOR, patience=PATIENCE_REDUCE, min_lr=MIN_LR, verbose=1),
            EarlyStopping(monitor='val_accuracy', patience=PATIENCE_EARLY, restore_best_weights=True, verbose=1)
        ]
        model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=cb, verbose=1)
        model.load_weights(model_path)

    # Save predictions
    print(f"Generating predictions...")
    val_preds = model.predict(val_ds, verbose=0)
    test_preds = model.predict(test_ds, verbose=0)
    np.save(val_pred_path, val_preds)
    np.save(test_pred_path, test_preds)

    acc = accuracy_score(y_true_flat, np.argmax(test_preds, axis=1))
    print(f"{model_name} Test Accuracy: {acc:.4f}")

    del model
    gc.collect()
    return acc

## Model Training

In [ ]:
train_and_save('EfficientNetV2M')


==================== TRAINING EfficientNetV2M ====================
214201816/214201816 [==============================] - 8s 0us/step
Epoch 1/30


I0000 00:00:1780855939.547529     193 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1780855950.399984     194 service.cc:152] XLA service 0x7df35cbd46c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780855950.400019     194 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1780855950.400023     194 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1780855950.846346     191 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1752/1752 [==============================] - ETA: 0s - loss: 0.8285 - accuracy: 0.7273
Epoch 1: val_accuracy improved from -inf to 0.79163, saving model to /kaggle/working/dermamnist_outputs/EfficientNetV2M_dermamnist.keras
1752/1752 [==============================] - 400s 136ms/step - loss: 0.8285 - accuracy: 0.7273 - val_loss: 0.5816 - val_accuracy: 0.7916 - lr: 1.0000e-04
Epoch 2/30
1752/1752 [==============================] - ETA: 0s - loss: 0.5903 - accuracy: 0.7988
Epoch 2: val_accuracy improved from 0.79163 to 0.79262, saving model to /kaggle/working/dermamnist_outputs/EfficientNetV2M_dermamnist.keras
1752/1752 [==============================] - 227s 129ms/step - loss: 0.5903 - accuracy: 0.7988 - val_loss: 0.6885 - val_accuracy: 0.7926 - lr: 1.0000e-04
Epoch 3/30
1752/1752 [==============================] - ETA: 0s - loss: 0.4305 - accuracy: 0.8490
Epoch 3: val_accuracy improved from 0.79262 to 0.81755, saving model to /kaggle/working/dermamnist_outputs/EfficientNetV2M_dermamnis

0.8807980049875311

In [ ]:
train_and_save('ConvNeXtBase')


==================== TRAINING ConvNeXtBase ====================
350926856/350926856 [==============================] - 13s 0us/step
Epoch 1/30
1752/1752 [==============================] - ETA: 0s - loss: 0.7207 - accuracy: 0.7415
Epoch 1: val_accuracy improved from -inf to 0.82054, saving model to /kaggle/working/dermamnist_outputs/ConvNeXtBase_dermamnist.keras
1752/1752 [==============================] - 561s 245ms/step - loss: 0.7207 - accuracy: 0.7415 - val_loss: 0.4875 - val_accuracy: 0.8205 - lr: 1.0000e-04
Epoch 2/30
1752/1752 [==============================] - ETA: 0s - loss: 0.3829 - accuracy: 0.8586
Epoch 2: val_accuracy improved from 0.82054 to 0.86042, saving model to /kaggle/working/dermamnist_outputs/ConvNeXtBase_dermamnist.keras
1752/1752 [==============================] - 268s 153ms/step - loss: 0.3829 - accuracy: 0.8586 - val_loss: 0.4051 - val_accuracy: 0.8604 - lr: 1.0000e-04
Epoch 3/30
1752/1752 [==============================] - ETA: 0s - loss: 0.1762 - accuracy: 0

0.8947630922693267

In [ ]:
train_and_save('ViT-Base')


==================== TRAINING ViT-Base ====================


config.json: 0.00B [00:00, ?B/s]

tf_model.h5:   0%|          | 0.00/347M [00:00<?, ?B/s]

All model checkpoint layers were used when initializing TFViTForImageClassification.

Some weights of TFViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier/kernel:0: found shape (768, 1000) in the checkpoint and (768, 7) in the model instantiated
- classifier/bias:0: found shape (1000,) in the checkpoint and (7,) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/30
1752/1752 [==============================] - ETA: 0s - loss: 0.5991 - accuracy: 0.7846
Epoch 1: val_accuracy improved from -inf to 0.84845, saving model to /kaggle/working/dermamnist_outputs/ViT-Base_dermamnist.keras
1752/1752 [==============================] - 237s 112ms/step - loss: 0.5991 - accuracy: 0.7846 - val_loss: 0.4326 - val_accuracy: 0.8485 - lr: 2.0000e-05
Epoch 2/30
1752/1752 [==============================] - ETA: 0s - loss: 0.3056 - accuracy: 0.8921
Epoch 2: val_accuracy improved from 0.84845 to 0.88036, saving model to /kaggle/working/dermamnist_outputs/ViT-Base_dermamnist.keras
1752/1752 [==============================] - 196s 112ms/step - loss: 0.3056 - accuracy: 0.8921 - val_loss: 0.3442 - val_accuracy: 0.8804 - lr: 2.0000e-05
Epoch 3/30
1752/1752 [==============================] - ETA: 0s - loss: 0.1282 - accuracy: 0.9553
Epoch 3: val_accuracy did not improve from 0.88036
1752/1752 [==============================] - 188s 108ms/step - loss: 0.1282 - accura

0.9057356608478803

In [ ]:
train_and_save('InceptionResNetV2')


==================== TRAINING InceptionResNetV2 ====================
219055592/219055592 [==============================] - 9s 0us/step
Epoch 1/30
1752/1752 [==============================] - ETA: 0s - loss: 0.8133 - accuracy: 0.7129
Epoch 1: val_accuracy improved from -inf to 0.70788, saving model to /kaggle/working/dermamnist_outputs/InceptionResNetV2_dermamnist.keras
1752/1752 [==============================] - 249s 111ms/step - loss: 0.8133 - accuracy: 0.7129 - val_loss: 5.8161 - val_accuracy: 0.7079 - lr: 1.0000e-04
Epoch 2/30
1752/1752 [==============================] - ETA: 0s - loss: 0.5719 - accuracy: 0.7959
Epoch 2: val_accuracy improved from 0.70788 to 0.73679, saving model to /kaggle/working/dermamnist_outputs/InceptionResNetV2_dermamnist.keras
1752/1752 [==============================] - 163s 93ms/step - loss: 0.5719 - accuracy: 0.7959 - val_loss: nan - val_accuracy: 0.7368 - lr: 1.0000e-04
Epoch 3/30
1752/1752 [==============================] - ETA: 0s - loss: 0.4291 - a

0.827431421446384